# ОКС: SHAP окончательной модели и итог нового эксперимента

Используется только окончательная модель нового запуска без отдельного индикатора отсутствия СРБ. Сам СРБ и другие разрешённые индикаторы сохранены. Объясняются реальные записи полной когорты; шкала и аддитивность проверяются относительно фактического выхода классификатора. Прежние результаты используются исключительно как архив для сопоставления до/после. Семейство не задаётся заранее.

In [ ]:
from pathlib import Path
import sys, os, json, hashlib, platform, importlib, datetime
PROJECT_ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
                     if (p / "studies/acs/configs/layout.json").is_file()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Откройте ноутбук внутри клонированного acs-research-projects")
sys.path.insert(0, str(PROJECT_ROOT / "src"))
from acs_research.paths import study_paths
P = study_paths(PROJECT_ROOT)
STUDY = P.study
OUT = P.run
OUT.mkdir(parents=True, exist_ok=True)
os.environ.setdefault("MPLCONFIGDIR", str(OUT / ".matplotlib"))
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
BASE_SEED = 2026
VERSIONS = {m: importlib.import_module(m).__version__ for m in
            ["numpy", "pandas", "sklearn", "optuna", "shap", "catboost", "xgboost", "nbclient"]}
print(platform.python_version(), "seed", BASE_SEED, VERSIONS)
def sha(path): return hashlib.sha256(Path(path).read_bytes()).hexdigest()
def savejson(obj, name):
    (OUT / name).write_text(json.dumps(obj, ensure_ascii=False, indent=2, default=str), encoding="utf-8")
def figsave(name):
    plt.tight_layout()
    plt.savefig(OUT / (name + ".png"), dpi=180, bbox_inches="tight")
    plt.savefig(OUT / (name + ".svg"), bbox_inches="tight")
    plt.show(); plt.close()


In [ ]:
import joblib, shap
from scipy.special import logit
meta=json.loads((OUT/"01_конфигурация.json").read_text())
protocol=json.loads((OUT/"02_протокол.json").read_text())
assert sha(OUT/"01_данные.csv")==meta["data_sha256"]==protocol["data_sha256"]
SHAP_MODEL_HASH=sha(OUT/"02_final_model.joblib")
for p,h in meta["source_sha256"].items():assert sha(PROJECT_ROOT/p)==h
artifact=joblib.load(OUT/"02_final_model.joblib")
assert artifact["run_id"]==protocol["run_id"]
FINAL=json.loads((OUT/"02_final_config.json").read_text())
D=pd.read_csv(OUT/"01_данные.csv");FEATURES=artifact["features"];X=D[FEATURES];y=D.hospital_death.to_numpy()
pre=artifact["preprocessor"];model=artifact["model"];family=artifact["family"]
assert list(model.classes_)==[0,1]
z=pre.transform(X);transformed_names=pre.get_feature_names_out().tolist()
assert "numeric__crp" in transformed_names and "missing__missingindicator_crp" not in transformed_names
sampling_audit=pd.read_csv(OUT/"02_аудит_балансировки.csv")
fit_audit=sampling_audit[(sampling_audit.seed==BASE_SEED+9999)&(sampling_audit.train_n==len(X))&(sampling_audit.strategy==artifact["config"]["strategy"])]
assert len(fit_audit)
fit_n=int(fit_audit.iloc[-1].selected_n)
p=model.predict_proba(z)[:,1];t=artifact["threshold"]
rng=np.random.default_rng(BASE_SEED);bg_ids=np.sort(rng.choice(len(X),min(100,len(X)),replace=False));background=z[bg_ids]
BACKGROUND_DESC="случайный реальный фон из 100 наблюдений, seed=2026"
if family=="LR":
    explainer=shap.LinearExplainer(model,shap.maskers.Independent(background,max_samples=len(background)))
    explanation=explainer(z);values=explanation.values;base=np.broadcast_to(explanation.base_values,(len(X),)).copy()
    output=model.decision_function(z);scale="log-odds смерти";perturbation="Independent masker, interventional"
else:
    if family=="RF":
        # Same RF explanation definition as the archived study; strict additivity is checked anew.
        # Background uses the fitted trees' training path counts.
        assert "missing__missingindicator_crp" not in transformed_names
        bg_ids=np.arange(len(X))
        explainer=shap.TreeExplainer(model,feature_perturbation="tree_path_dependent",model_output="raw")
        BACKGROUND_DESC="неявный фон: частоты путей обучающих реальных записей в деревьях; частоты учитывают бутстреп и выбранную стратегию весов/отбора"
        np.savez_compressed(OUT/"03_background_счетчики_деревьев.npz",node_sample_weight=explainer.model.node_sample_weight,children_left=explainer.model.children_left,children_right=explainer.model.children_right,training_row_id=D.row_id.to_numpy())
    else:
        explainer=shap.TreeExplainer(model,data=background,feature_perturbation="interventional",model_output="raw")
    explanation=explainer(z)
    values=explanation.values;base=explanation.base_values
    if values.ndim==3:
        values=values[:,:,1];base=np.asarray(base)[:,1]
    base=np.broadcast_to(base,(len(X),)).copy()
    if family=="RF":output=p;scale="вероятность смерти"
    elif family=="CatBoost":output=model.predict(z,prediction_type="RawFormulaVal");scale="log-odds смерти"
    else:output=model.predict(z,output_margin=True);scale="log-odds смерти"
    perturbation="tree_path_dependent, implicit real training path counts, model_output=raw" if family=="RF" else "interventional, real background n=100, model_output=raw"
assert values.shape==z.shape
error=float(np.max(abs(base+values.sum(axis=1)-output)))
assert np.isfinite(values).all() and error<1e-5, error
np.savez_compressed(OUT/"03_shap_values.npz",run_id=np.array([artifact["run_id"]]),model_sha256=np.array([SHAP_MODEL_HASH]),values=values,base_values=base,model_output=output,p_death=p,row_id=D.row_id.to_numpy(),group_id=D.group_id.to_numpy(),background_row_id=D.row_id.iloc[bg_ids].to_numpy(),transformed_feature_names=np.array(transformed_names))
print("Chosen model",family,"scale",scale,"additivity max error",error)

savejson({"run_id":artifact["run_id"],"family":family,"scale":scale,"max_additivity_error":error,"tolerance":1e-5,"model_output":"predict_proba death" if family=="RF" else "decision margin death","n":len(X)},"03_диагностика_аддитивности.json")
if family=="RF":savejson({"run_id":artifact["run_id"],"mode":perturbation,"max_additivity_error":error,"n":len(X)},"03_диагностика_tree_path.json")


## Mapping и клиническая визуализация
Масштабированные входы LR не показываются как клинические единицы: при визуализации используются исходные характеристики. Флаги неполноты обозначают регистрацию измерений, а не отрицательный диагноз. Режим SHAP зависит от определения фона и обработки корреляций; ранги объясняют конкретный выход этой модели.

In [ ]:
mapping=[];display_keys=[]
for name in transformed_names:
    suffix=name.split("__",1)[1]
    if name.startswith("numeric__"):
        feature=suffix;key=feature;kind="clinical"
    elif name.startswith("categorical__"):
        feature=suffix.rsplit("_",1)[0];key=feature;kind="clinical"
    else:
        feature=suffix.replace("missingindicator_","");key="missing_"+feature;kind="missing_indicator"
    assert feature in FEATURES,(name,feature)
    mapping.append(dict(transformed_feature=name,source_feature=feature,display_key=key,kind=kind))
    if key not in display_keys:display_keys.append(key)
mapping=pd.DataFrame(mapping)
assert not ((mapping.source_feature=="crp")&(mapping.kind=="missing_indicator")).any()
assert "crp" in display_keys and "missing_crp" not in display_keys
mapping.to_csv(OUT/"03_mapping_признаков.csv",index=False)
groupvalues=np.column_stack([values[:,np.flatnonzero(mapping.display_key==key)].sum(axis=1) for key in display_keys])
display_data=np.column_stack([X[key].to_numpy() if key in FEATURES else X[key.removeprefix("missing_")].isna().to_numpy().astype(int) for key in display_keys])
labels=[meta["labels"][key] if key in FEATURES else "Пропуск: "+meta["labels"][key.removeprefix("missing_")] for key in display_keys]
assert np.max(abs(base+groupvalues.sum(axis=1)-output))<1e-5
clinical_explanation=shap.Explanation(values=groupvalues,base_values=base,data=display_data,feature_names=labels)
importance=pd.DataFrame(dict(key=display_keys,label=labels,mean_abs_SHAP=np.mean(abs(groupvalues),axis=0),mean_signed_SHAP=np.mean(groupvalues,axis=0),kind=["clinical" if k in FEATURES else "missing_indicator" for k in display_keys])).sort_values("mean_abs_SHAP",ascending=False).reset_index(drop=True)
importance["rank"]=np.arange(1,len(importance)+1);importance.to_csv(OUT/"03_важности_SHAP.csv",index=False)
pd.DataFrame(dict(feature=transformed_names,mean_abs_SHAP=np.mean(abs(values),axis=0))).sort_values("mean_abs_SHAP",ascending=False).to_csv(OUT/"03_важности_преобразованных_признаков.csv",index=False)
np.savez_compressed(OUT/"03_SHAP_клинические_группы.npz",run_id=np.array([artifact["run_id"]]),model_sha256=np.array([SHAP_MODEL_HASH]),values=groupvalues,base_values=base,display_data=display_data,keys=np.array(display_keys),labels=np.array(labels),row_id=D.row_id.to_numpy())
display(importance)
shap.plots.bar(clinical_explanation,max_display=20,show=False)
plt.title(f"{family}: средняя величина вкладов, {scale}");figsave("03_SHAP_bar")
shap.plots.beeswarm(clinical_explanation,max_display=20,show=False,plot_size=(12,9))
plt.title(f"{family}: распределение вкладов, {scale}");figsave("03_SHAP_beeswarm")


In [ ]:
top=importance.loc[importance.kind=="clinical","key"].head(5).tolist()
DEPENDENCIES=list(dict.fromkeys(top+["tag","ms","crp"]))
for feature in DEPENDENCIES:
    j=display_keys.index(feature);a=X[feature].to_numpy();v=groupvalues[:,j];valid=np.isfinite(a)
    plt.figure(figsize=(7,4.5))
    plt.scatter(a[valid],v[valid],c=y[valid],cmap="coolwarm",s=18,alpha=.5)
    plt.axhline(0,color="grey",ls="--",lw=1);plt.xlabel(meta["labels"][feature]);plt.ylabel("SHAP: "+scale)
    plt.title(f"{family}: {meta['labels'][feature]}")
    if (~valid).any():plt.text(.02,.97,f"Пропуск: {(~valid).sum()} записей; средний вклад {np.mean(v[~valid]):.4f}",transform=plt.gca().transAxes,va="top")
    plt.colorbar(label="Госпитальная смерть 1 / выжил 0");figsave("03_зависимость_"+feature)
examples=[]
def add_example(reason,ids):
    if len(ids):
        i=int(ids[0]);examples.append(dict(reason=reason,row_id=int(D.row_id.iloc[i]),position=i,y_true=int(y[i]),p_death=float(p[i]),threshold=float(t),predicted_death=int(p[i]>=t),base=float(base[i]),model_output=float(output[i]),additivity_error=float(abs(base[i]+groupvalues[i].sum()-output[i]))))
add_example("Первый умерший",np.flatnonzero(y==1))
add_example("Первый выживший",np.flatnonzero(y==0))
add_example("Ближайший к порогу",[np.argmin(abs(p-t))])
add_example("Первый FN на полной когорте",np.flatnonzero((y==1)&(p<t)))
add_example("Первый FP на полной когорте",np.flatnonzero((y==0)&(p>=t)))
exampletable=pd.DataFrame(examples);exampletable.to_csv(OUT/"03_индивидуальные_примеры.csv",index=False)
for e in examples:
    shap.plots.waterfall(clinical_explanation[e["position"]],max_display=14,show=False)
    plt.title(f"Вклады в прогноз смерти, row_id={e['row_id']}: {e['reason']}\np={e['p_death']:.3f}; исход={e['y_true']}; {scale}",fontsize=11)
    figsave("03_waterfall_"+str(e["row_id"]))
shapconfig={"run_id":artifact["run_id"],"family":family,"versions":VERSIONS,"model_sha256":sha(OUT/"02_final_model.joblib"),"data_sha256":meta["data_sha256"],"n_explained":len(X),"input_cohort_n":len(X),"classifier_training_n":fit_n,"background_n":len(bg_ids),"background_description":BACKGROUND_DESC,"background_row_id":D.row_id.iloc[bg_ids].tolist(),"seed":BASE_SEED,"scale":scale,"perturbation":perturbation,"max_additivity_error":error,"tolerance":1e-5,"population":"all real input cohort records; classifier training subset is specified separately","grouping":"sums of transformed one-hot contributions; not Shapley for a grouped game","examples_rule":"first death; first survival; closest to threshold; first full-cohort FN and FP if present"}
savejson(shapconfig,"03_конфигурация_SHAP.json")
display(exampletable)


## Независимые прогнозы в рисунках для статьи
Матрицы ошибок и калибровка ниже строятся по сохранённым внешним OOF, а не по обучающим прогнозам финального артефакта. На калибровочных графиках подписаны число смертей/наблюдений в каждой группе для каждого повторения. Фигуры не включают SHAP других моделей.

In [ ]:
provenance=json.loads((OUT/"02_происхождение_результатов.json").read_text())
assert SHAP_MODEL_HASH==sha(OUT/"02_final_model.joblib")==provenance["model_sha256"]
assert meta["data_sha256"]==provenance["data_sha256"]
print("Completed matching notebook 02 results verified",provenance["run_id"])
# Additional presentation of independent CV predictions; no additional SHAP models.
from sklearn.metrics import confusion_matrix
external=pd.read_csv(OUT/"02_oof_predictions.csv")
cal=pd.read_csv(OUT/"02_калибровка.csv")
fig,axes=plt.subplots(1,3,figsize=(14,4))
for r,ax in enumerate(axes):
    g=external[(external.family=="selector")&(external.repeat==r)]
    cm=confusion_matrix(g.y_true,g.p_death>=g.threshold,labels=[0,1])
    ax.imshow(cm,cmap="Blues")
    for i in range(2):
        for j in range(2):ax.text(j,i,str(cm[i,j]),ha="center",va="center",color="white" if cm[i,j]>cm.max()/2 else "black",fontsize=15)
    ax.set_xticks([0,1],["Выжил","Смерть"]);ax.set_yticks([0,1],["Выжил","Смерть"])
    ax.set(xlabel="Прогноз",ylabel="Исход",title=f"Selector, OOF-повторение {r+1}")
figsave("03_selector_матрицы_ошибок")
for f in ["LR","RF","CatBoost","XGBoost","selector"]:
    fig,axes=plt.subplots(1,3,figsize=(15,7))
    for r,ax in enumerate(axes):
        g=cal[(cal.family==f)&(cal.repeat==r)]
        ax.plot(g.mean_prediction,g.event_rate,"o-",color="#35658c")
        counts=[[k+1,int(v.n),int(v.deaths),f"{v.mean_prediction:.4f}",f"{v.event_rate:.4f}"] for k,(_,v) in enumerate(g.iterrows())]
        table=ax.table(cellText=counts,colLabels=["Группа","n","Смертей","Прогноз","Частота"],bbox=[0,-.72,1,.5],cellLoc="center")
        table.auto_set_font_size(False);table.set_fontsize(9)
        ax.plot([0,1],[0,1],"--",color="grey",lw=1)
        limit=min(1,max(cal.loc[cal.family==f,"mean_prediction"].max(),cal.loc[cal.family==f,"event_rate"].max())*1.15)
        ax.set(xlabel="Средний прогноз смерти",ylabel="Частота смерти",title=f"{f}, повторение {r+1}",xlim=(0,limit),ylim=(0,limit))
    figsave("03_калибровка_численности_"+f)
# The exact 95% intervals remain conditional on fixed CV predictions.
flow=[("906 госпитализаций\n43 смерти; 892 группы",.11), ("3 × 3 внешних блока\n3 внутренних блока",.36),("4 семейства, Optuna\n50 trials на study",.61),("Selector OOF\nОдна финальная модель + SHAP",.86)]
fig,ax=plt.subplots(figsize=(14,3));ax.axis("off")
for j,(label,x) in enumerate(flow):
    ax.text(x,.5,label,ha="center",va="center",transform=ax.transAxes,bbox=dict(boxstyle="round,pad=.7",facecolor="#e8eff4",edgecolor="#35658c"),fontsize=11)
    if j<len(flow)-1:ax.annotate("",xy=(x+.17,.5),xytext=(x+.1,.5),xycoords="axes fraction",arrowprops=dict(arrowstyle="->",color="#35658c"))
figsave("03_схема_исследования")


## Итог и проверка

SHAP относится к модели на полной когорте; независимое качество относится к selector.
Индивидуальные waterfall, beeswarm, scatter и NPZ остаются локально и исключены из Git.
Публичные результаты исходного завершённого запуска сохранены отдельно в results/.


In [ ]:
summary=pd.read_csv(OUT/"02_сравнение_моделей.csv").set_index("name")
display(summary.loc[["LR","RF","CatBoost","XGBoost","selector"], ["AP","ROC_AUC","Brier"]])
display(importance.head(10))
from acs_research.validation import validate_run
audit=validate_run(OUT)
savejson(audit, "repository_validation.json")
print("Проверка сохранённых результатов:", audit)
